# 01 — Dataset Audit (Stage S0.2)

**DRIVE** project · Primary dataset: Alibaba SSD Open Data (dcbrain), 2018–2019.

This notebook produces exactly the audit items required by the implementation plan
(handover §5) **per source**, feeding the S0.3 decision gate:

1. Files / sizes / date ranges
2. Schema per release
3. Unique models, unique drives, total daily rows vs unique drives
4. SSD classification method + validation
5. SSD **failure-event count**  ← S0.3 gate: ≥ ~100 events over 2018–2019
6. Failure semantics (what counts as one event)
7. Infrastructure-field availability/missingness (`rack_id`, `node_id`)
8. SMART coverage
9. Duplicates
10. Chronological-experiment feasibility
11. Cross-release join test (`disk_id`/model overlap with Tianchi daily-SMART release)


In [ ]:
import zipfile
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)

RAW = Path("..") / "data" / "raw" / "alibaba_core"
FILES = {
    "failure_tag": "ssd_failure_tag.csv.zip",
    "location": "location_info_of_ssd.csv.zip",
    "smart": "smart_log_20191231.csv.zip",
}
for name, fname in FILES.items():
    p = RAW / fname
    print(f"{name:12} exists={p.exists()}  size={p.stat().st_size / 1e6:.1f} MB" if p.exists() else f"{name:12} MISSING")

## 1. Files, sizes, date ranges

- [ ] Expected three files (see `FILES` above)
- [ ] `failure_time` date range in the failure-tag file — must cover 2018-01 → 2019-12
- [ ] SMART snapshot date is a single day (2019-12-31) — record as a limitation


## 2. Schema per release

Verify columns against the official readme:

| File | Expected key columns |
|---|---|
| failure_tag | model, failure_time, failure, app, machine_room_id, rack_id, node_id, disk_id, r_/n_ SMART attrs (39 cols) |
| location | app, model, rack_id, node_id, disk_id, slot_id (6 cols) |
| smart | model, disk_id, ds, r_*/n_* (105 cols) |


In [ ]:
# TODO: read the CSV header of each zip member with nrows=5 and print `df.shape` and
# `df.columns.tolist()`; confirm against the table above. Do NOT load full data.
def peek(fname, n_rows=5):
    with zipfile.ZipFile(RAW / fname) as z:
        member = [m for m in z.namelist() if not m.endswith('/\')][0]
        with z.open(member) as f:
            df = pd.read_csv(f, nrows=n_rows)
    return df

# df_ft = peek(FILES['failure_tag']);  df_ft.head()

## 3. Uniques: models, drives, rows vs drives

- [ ] unique `model` count (expect ~11)
- [ ] unique `disk_id` count (expect ~1M across sources)
- [ ] total rows in failure_tag vs unique drives in failure_tag
      (row/drive confusion check — handover mistake list)

## 4. SSD classification + validation

- [ ] All drives in this dataset are SSDs by construction (verify via model strings)
- [ ] Spot-check 10 random models manually; document the classification rule
      in `src/drive/preprocessing/identify_ssds.py` (S1)


## 5. ⛔ Gate check: SSD failure-event count (S0.3)

Count rows where `failure == 1` in `ssd_failure_tag.csv.zip`.

- **Gate:** ≥ ~100 failure events spread over 2018–2019
- Also: events per model, per year — must be spread, not one burst

> This single number decides whether the whole decision-engine experiment is feasible.


In [ ]:
# TODO: count failure==1 rows, events per model, events per year-month.
# Chunked read (usecols=['model','failure_time','failure','disk_id','rack_id','node_id']).


## 6. Failure semantics

- [ ] One row per failure event? Can one drive appear multiple times (repairable/repeat tickets)?
- [ ] Does the healthy population (failure==0) carry locations too (machine_room_id present)?
- [ ] Define the canonical **one event per drive/event** rule before S1


## 7. Infrastructure fields availability & missingness

- [ ] % missing `rack_id` / `node_id` in failure_tag and location
- [ ] Drives with location but no failure ticket, and vice versa
- [ ] Group sizes: SSDs per rack, per node — enough for smoothing (Gamma-Poisson) to be meaningful


## 8. SMART coverage

- [ ] Unique drives in the one-day SMART snapshot vs drives in failure_tag/location
- [ ] SMART attribute coverage across models (which r_/n_ columns are populated per model)


## 9. Duplicates

- [ ] Duplicate keys: (disk_id) in location; (disk_id, failure_time) in failure_tag
- [ ] Duplicate rows in the SMART snapshot


## 10. Chronological-experiment feasibility

DRIVE needs: a decision date t, history ≤ t, evaluation window > t.

- [ ] Failure-time distribution over 2018–2019 — enough failures before t **and** after t
- [ ] Candidate decision dates (e.g., mid-2019) with ≥ 100 failure events in the
      post-t window
- [ ] Note: rows > drives, so the healthy population must be reconstructed from
      location + SMART snapshot + failure tags (record the chosen exposure model)


## 11. Cross-release join test (Tianchi supplementary)

 deferred until the Tianchi daily-SMART release is downloaded.

- [ ] Overlap of `disk_id`/model between Tianchi daily-SMART (6 models) and the dcbrain files
- [ ] If the join works: real drive-day exposure + ML baseline upgrade apply


## Verdict (S0.3 decision gate)

Fill in after all sections:

| Item | Result | Gate |
|---|---|---|
| Failure events (total / spread) | _pending_ | ≥ ~100 ✅/❌ |
| Rack/node identifiers reliable | _pending_ | ✅/❌ |
| Files joinable on disk_id | _pending_ | ✅/❌ |
| Chronological experiment feasible | _pending_ | ✅/❌ |
| **Alibaba primary verdict** | | |

Backblaze role (full second experiment / recency supplement / drop) is decided by the
separate Backblaze audit — start with 2–3 quarters only after this gate passes.
